# Step 3 — Sliding Window Segmentation + Data Split

**RESS 2025 — GAN-Conformal-RUL**

Turns the per-bearing feature matrices + stage labels from Step 2 into model-ready windows.

| Parameter | Value | Rationale |
|---|---|---|
| Window size | 32 | Power of 2, GAN-friendly |
| Stride | 1 | Max data from short bearings |
| RUL target | Normalized 0–1 per bearing | Matches XJTU-SY literature (lifetimes span 42–2538 min) |
| Split level | **Bearing** (not window) | Stride-1 windows overlap 31/32 → random split would leak and break the conformal exchangeability assumption |
| Per fold | 9 train / 1 val / 2 cal / 3 test | Calibration bearings never seen in training |

## 1. Setup

In [ ]:
import os, sys, shutil

# Reset cwd first (avoids getcwd errors if a prior run deleted the folder)
os.chdir("/content")

REPO_PATH = "/content/RESS_2025_GAN_Conformal_RUL"
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f"{REPO_PATH}/src")

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = f"{REPO_PATH}/figures"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Setup OK")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import XJTUSYLoader
from health_indicator import HealthIndicatorPipeline
from windowing import (make_windows, build_folds, prepare_fold,
                       prepare_all_folds, WINDOW_SIZE, STRIDE)

# ── Stage color code (SAME across every figure in the paper) ──────────
STAGE_COLORS = {1: '#2ca02c', 2: '#ff9800', 3: '#e34948'}
STAGE_LABELS = {1: 'S1 Healthy', 2: 'S2 Early Deg.', 3: 'S3 Near-Failure'}

print(f"Window size: {WINDOW_SIZE} | Stride: {STRIDE}")

## 2. Rebuild Step 2 results (HI + stage labels)

In [ ]:
# Auto-locate XJTU-SY on Drive
CANDIDATES = [
    '/content/drive/MyDrive/XJTU-SY',
    '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
    '/content/drive/MyDrive/data/XJTU-SY',
]
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"XJTU-SY not found. Tried: {CANDIDATES}")
print(f"Data root: {DATA_ROOT}")

loader = XJTUSYLoader(DATA_ROOT)
all_data = loader.load_all()

pipeline = HealthIndicatorPipeline()
results = pipeline.process_all(all_data, verbose=True)

## 3. Windowing — single bearing sanity check

Verify the shapes and the causal labeling convention (each window is labeled by its **last** time step, so the model never sees the future).

In [ ]:
bid = 'Bearing2_1'
r = results[bid]
X, y_rul, y_stage = make_windows(r['features'], r['stage_labels'])

print(f"{bid}: {r['features'].shape[0]} recordings -> {len(X)} windows")
print(f"  X       : {X.shape}   (n_windows, window_size, n_features)")
print(f"  y_rul   : {y_rul.shape}  range [{y_rul.min():.3f}, {y_rul.max():.3f}]")
print(f"  y_stage : {y_stage.shape}  values {sorted(set(y_stage.tolist()))}  (0-indexed)")
print()
print("Windows per bearing (window=32):")
for b, rr in results.items():
    n_rec = rr['features'].shape[0]
    n_win = max(0, n_rec - WINDOW_SIZE + 1)
    warn = '  <-- very few' if 0 < n_win < 30 else ('  <-- TOO SHORT' if n_win == 0 else '')
    print(f"  {b:14s} {n_rec:5d} rec -> {n_win:5d} windows{warn}")

In [ ]:
# RUL target + stage label over the bearing's life
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(y_rul, color='#1f77b4', lw=1.5)
axes[0].set_ylabel('RUL (normalized)')
axes[0].set_title(f'RUL target and stage labels per window — {bid.replace("Bearing","Bearing ").replace("_","-")}')
axes[0].grid(alpha=0.3)

for s in (1, 2, 3):
    m = (y_stage == s - 1)
    axes[1].scatter(np.where(m)[0], np.full(m.sum(), s),
                    color=STAGE_COLORS[s], s=4, label=STAGE_LABELS[s])
axes[1].set_yticks([1, 2, 3])
axes[1].set_yticklabels([STAGE_LABELS[s] for s in (1, 2, 3)])
axes[1].set_xlabel('Window index')
axes[1].legend(loc='center left', fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Fold construction + leakage check

**Critical:** no bearing may appear in more than one split within a fold. If it did, near-identical overlapping windows would sit in both train and calibration, invalidating the conformal coverage guarantee.

In [ ]:
folds = build_folds()

print("=== LEAKAGE CHECK ===")
ok = True
for f in folds:
    allb = f['test'] + f['cal'] + f['val'] + f['train']
    if len(allb) != len(set(allb)):
        dup = [b for b in set(allb) if allb.count(b) > 1]
        print(f"  Fold {f['fold']}: LEAK — {dup}"); ok = False
    if len(allb) != 15:
        print(f"  Fold {f['fold']}: {len(allb)}/15 bearings assigned"); ok = False
print("  PASS — no overlap, all 15 bearings assigned in every fold\n" if ok else "  FAIL\n")

print("=== FOLD COMPOSITION ===")
short = lambda bs: [b.replace('Bearing','').replace('_','-') for b in bs]
for f in folds:
    print(f"  Fold {f['fold']}: test={short(f['test'])}  cal={short(f['cal'])}  "
          f"val={short(f['val'])}  train={len(f['train'])} bearings")

## 5. Prepare all folds

In [ ]:
fold_data = prepare_all_folds(results, verbose=True)

In [ ]:
# Summary table — watch calibration size (need >~200 for a stable quantile)
rows = []
for d, f in zip(fold_data, folds):
    rows.append({
        'fold': d['fold'],
        'train': len(d['X_train']), 'val': len(d['X_val']),
        'cal': len(d['X_cal']), 'test': len(d['X_test']),
        'cal_bearings': ', '.join(short(f['cal'])),
        'S3_in_cal': int((d['y_stage_cal'] == 2).sum()),
    })
df_folds = pd.DataFrame(rows)
print(df_folds.to_string(index=False))

low = df_folds[df_folds['cal'] < 200]
print("\nAll folds have adequate calibration size (>200 windows)." if low.empty
      else f"\nWARNING — folds with <200 cal windows: {low['fold'].tolist()}")

## 6. Class balance across splits

This is the imbalance the conditional GAN (C1) exists to correct — Stage 3 is the scarce class in the training pool.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=True)
splits = ['train', 'val', 'cal', 'test']

for ax, d in zip(axes, fold_data):
    bottoms = np.zeros(len(splits))
    for s in (1, 2, 3):
        pcts = []
        for sp in splits:
            ys = d[f'y_stage_{sp}']
            pcts.append(100 * (ys == s - 1).sum() / max(len(ys), 1))
        pcts = np.array(pcts)
        ax.bar(splits, pcts, bottom=bottoms, color=STAGE_COLORS[s],
               alpha=0.85, label=STAGE_LABELS[s] if d['fold'] == 1 else None)
        bottoms += pcts
    ax.set_title(f"Fold {d['fold']}", fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(0, 100)

axes[0].set_ylabel('Percentage of windows (%)')
fig.legend(loc='upper center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.06))
plt.suptitle('Stage distribution across splits, per fold', y=1.12, fontsize=12)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_class_balance_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Window counts per split, per fold
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(fold_data)); width = 0.2
cols = {'train': '#4c72b0', 'val': '#8172b2', 'cal': '#dd8452', 'test': '#55a868'}

for i, sp in enumerate(splits):
    counts = [len(d[f'X_{sp}']) for d in fold_data]
    b = ax.bar(x + (i - 1.5) * width, counts, width, label=sp, color=cols[sp], alpha=0.9)
    ax.bar_label(b, fontsize=7, padding=1)

ax.axhline(200, color='red', ls='--', lw=1,
           label='min. cal. size (200)')
ax.set_xticks(x); ax.set_xticklabels([f"Fold {d['fold']}" for d in fold_data])
ax.set_ylabel('Number of windows')
ax.set_title(f'Window counts per split (window={WINDOW_SIZE}, stride={STRIDE})')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_window_counts_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Verify the scaler (fitted on TRAIN only — no leakage)

In [ ]:
d = fold_data[0]
print("Fold 1 — after scaling (mu/sigma fitted on TRAIN windows only):\n")
print(f"{'split':<7} {'mean':>9} {'std':>9}   (train should be ~0 / ~1;")
print(f"{'':<7} {'':>9} {'':>9}    others may deviate — that is correct)")
print("-" * 40)
for sp in splits:
    X = d[f'X_{sp}']
    if len(X):
        print(f"{sp:<7} {X.mean():>9.4f} {X.std():>9.4f}")

print(f"\nFinal shapes — X: {d['X_train'].shape}")
print(f"RUL range: [{d['y_rul_train'].min():.3f}, {d['y_rul_train'].max():.3f}]")
print("\nReady for Step 4 (conditional GAN).")

## Decisions recorded for the paper

- **Window = 32, stride = 1.** Power of two for GAN convolutions; stride 1 maximizes samples from short-lived bearings.
- **RUL normalized 0–1 per bearing.** Raw lifetimes span 42–2538 min; an absolute target would let long-lived bearings dominate the loss. Also makes RMSE comparable to published XJTU-SY baselines.
- **Causal labeling.** Each window takes the RUL and stage of its *last* time step — no future information leaks into the input.
- **Bearing-level splitting.** With stride 1, adjacent windows share 31/32 time steps. A random window-level split would place near-duplicates in both train and calibration, breaking the exchangeability assumption that the conformal coverage guarantee depends on. Holding out whole bearings makes the guarantee genuine rather than nominal.
- **Scaler fitted on training windows only**, then applied to val/cal/test.